#                                          Naive Bayes on Titanic Dataset

### 0. Download required libraries 


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report

### 1. Load Titanic dataset


In [2]:
df = pd.read_csv("titanic_train.csv")

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### 2.Keep only useful columns


In [4]:
df = df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Embarked']]
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
0,0,3,male,22.0,1,0,S
1,1,1,female,38.0,1,0,C
2,1,3,female,26.0,0,0,S
3,1,1,female,35.0,1,0,S
4,0,3,male,35.0,0,0,S


### 3. Handling missing values

In [5]:
missing_df = df.isnull().sum().to_frame(name='missing_values')
missing_df

,missing_values
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Embarked,2


In [6]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [7]:
df.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Embarked    0
dtype: int64

### 4. Convert ALL numeric columns into categories (bins)

In [8]:
df['Pclass'] = df['Pclass'].astype('category')


In [9]:
df['Age'] = pd.cut(df['Age'],
                   bins=[0, 12, 18, 35, 50, 80],
                   labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'])

In [10]:
df['SibSp'] = df['SibSp'].astype('category')

In [11]:
df['Parch'] = df['Parch'].astype('category')

### 5. Encode ALL columns using LabelEncoder

In [12]:
encoder = LabelEncoder()

for col in df.columns:
    df[col] = encoder.fit_transform(df[col])

### 6. Split data

In [13]:
X = df.drop('Survived', axis=1)
y = df['Survived']

In [14]:
X.head()

,Pclass,Sex,Age,SibSp,Parch,Embarked
0,2,1,4,1,0,2
1,0,0,0,1,0,0
2,2,0,4,0,0,2
3,0,0,4,1,0,2
4,2,1,4,0,0,2


In [15]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### 7. Train Categorical Naive Bayes

In [17]:
model = CategoricalNB()

In [18]:
model.fit(X_train, y_train)

,"alpha alpha: float, default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"min_categories min_categories: int or array-like of shape (n_features,), default=NoneMinimum number of categories per feature.- integer: Sets the minimum number of categories per feature to `n_categories` for each features.- array-like: shape (n_features,) where `n_categories[i]` holds the minimum number of categories for the ith column of the input.- None (default): Determines the number of categories automatically from the training data... versionadded:: 0.24",None


### 8. Prediction & Evaluation

In [19]:
preds = model.predict(X_test)

In [20]:
# Generate report as a dataframe instead of text

report_dict = classification_report(y_test, preds, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose().round(4)

In [21]:
df_report

,precision,recall,f1-score,support
0,0.7817,0.8284,0.8043,134.0000
1,0.7160,0.6517,0.6824,89.0000
accuracy,0.7578,0.7578,0.7578,0.7578
macro avg,0.7489,0.7400,0.7434,223.0000
weighted avg,0.7555,0.7578,0.7557,223.0000
